# NB12 — Final Evidence Consolidation

**Dissertation:** Explainable and Trustworthy Multimodal Deep Learning for Predictive Maintenance of Industrial Assets  
**Student:** 2023AA05069 | AIMLCZG628T | BITS Pilani  
**Notebook role:** Read-only consolidation of all final evidence. No model training. No test data access.  
**Environment:** Local

---

## Purpose

This notebook assembles the four final evidence artefacts (E53–E56) from the completed NB10 and NB11 outputs:

1. **E53** — Final model comparison table (validation + official test)
2. **E54** — Research question evidence table
3. **E55** — Final limitations table
4. **E56** — Final claims and qualifications

No model is trained or loaded. No test data is read. All numbers are read from saved CSV and JSON files.

## Section 0 — Setup

In [1]:
import os
import json
import pandas as pd
import numpy as np

BASE     = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
RV_DIR   = os.path.join(BASE, 'reports', 'final_validation')
TEST_DIR = os.path.join(BASE, 'reports', 'final_test')
MID_DIR  = os.path.join(BASE, 'reports', 'metrics')
OUT_DIR  = os.path.join(BASE, 'reports', 'final_summary')

os.makedirs(OUT_DIR, exist_ok=True)

# Verify all source files exist before reading anything
required = {
    'repeated_validation_summary':  os.path.join(RV_DIR,   'repeated_validation_summary_fd001.csv'),
    'repeated_validation_splits':   os.path.join(RV_DIR,   'repeated_validation_split_metrics_fd001.csv'),
    'fusion_comparison':            os.path.join(RV_DIR,   'repeated_validation_fusion_comparison_fd001.csv'),
    'cycle_index_ablation':         os.path.join(RV_DIR,   'cycle_index_ablation_fd001.csv'),
    'test_metrics':                 os.path.join(TEST_DIR, 'final_test_endpoint_metrics_fd001.csv'),
    'epoch_selection':              os.path.join(TEST_DIR, 'final_epoch_selection_fd001.csv'),
    'midsem_comparison':            os.path.join(MID_DIR,  'multiview_vs_baselines_comparison_fd001.csv'),
}

for label, path in required.items():
    assert os.path.isfile(path), f'Missing source file: {label} -> {path}'

print('All source files present.')
for label, path in required.items():
    print(f'  {label}: {os.path.relpath(path, BASE)}')

All source files present.
  repeated_validation_summary: reports/final_validation/repeated_validation_summary_fd001.csv
  repeated_validation_splits: reports/final_validation/repeated_validation_split_metrics_fd001.csv
  fusion_comparison: reports/final_validation/repeated_validation_fusion_comparison_fd001.csv
  cycle_index_ablation: reports/final_validation/cycle_index_ablation_fd001.csv
  test_metrics: reports/final_test/final_test_endpoint_metrics_fd001.csv
  epoch_selection: reports/final_test/final_epoch_selection_fd001.csv
  midsem_comparison: reports/metrics/multiview_vs_baselines_comparison_fd001.csv


## Section 1 — Load Source Artefacts

In [2]:
rv_summary  = pd.read_csv(required['repeated_validation_summary'])
rv_splits   = pd.read_csv(required['repeated_validation_splits'])
fusion_comp = pd.read_csv(required['fusion_comparison'])
ablation    = pd.read_csv(required['cycle_index_ablation'])
test_met    = pd.read_csv(required['test_metrics'])
epoch_sel   = pd.read_csv(required['epoch_selection'])
midsem      = pd.read_csv(required['midsem_comparison'])

print('Loaded:')
print(f'  rv_summary:  {rv_summary.shape}')
print(f'  rv_splits:   {rv_splits.shape}')
print(f'  fusion_comp: {fusion_comp.shape}')
print(f'  ablation:    {ablation.shape}')
print(f'  test_met:    {test_met.shape}')
print(f'  epoch_sel:   {epoch_sel.shape}')
print(f'  midsem:      {midsem.shape}')

Loaded:
  rv_summary:  (4, 16)
  rv_splits:   (12, 9)
  fusion_comp: (3, 11)
  ablation:    (3, 13)
  test_met:    (4, 12)
  epoch_sel:   (3, 7)
  midsem:      (5, 6)


## Section 2 — E53: Final Model Comparison Table

This table combines:
- Mid-semester seed-42 window-aligned validation results (NB06, historical)
- Final-protocol repeated-validation means across seeds 21, 42, 84 (NB10)
- Official FD001 endpoint test results (NB11)

The three result sets are not directly interchangeable — they differ in evaluation population.
The table documents all three for traceability.

In [3]:
# ── Mid-semester seed-42 results (NB06 historical, window-aligned validation) ─
# CNN1D included for completeness; it was not carried into final-protocol runs
midsem_rows = []
for _, row in midsem.iterrows():
    midsem_rows.append({
        'model':             row['model'],
        'midsem_val_rmse':   round(row['validation_rmse'], 4),
        'midsem_val_mae':    round(row['validation_mae'],  4),
        'midsem_val_r2':     round(row['validation_r2'],   4),
    })
midsem_df = pd.DataFrame(midsem_rows)

# ── Repeated-validation summary (NB10 final protocol) ─────────────────────────
rv_rows = []
for _, row in rv_summary.iterrows():
    rv_rows.append({
        'model':          row['model'],
        'rv_mean_rmse':   round(row['mean_rmse'], 4),
        'rv_std_rmse':    round(row['std_rmse'],  4),
        'rv_mean_mae':    round(row['mean_mae'],  4),
        'rv_mean_r2':     round(row['mean_r2'],   4),
        'rv_mean_bias':   round(row['mean_error_mean'], 4),
    })
rv_df = pd.DataFrame(rv_rows)

# ── Official test results (NB11) ──────────────────────────────────────────────
test_rows = []
for _, row in test_met.iterrows():
    test_rows.append({
        'model':          row['model'],
        'test_rmse':      round(row['rmse'],       4),
        'test_mae':       round(row['mae'],         4),
        'test_r2':        round(row['r2'],          4),
        'test_mean_bias': round(row['mean_error'],  4),
    })
test_df = pd.DataFrame(test_rows)

# ── Model ordering (by official test RMSE) ────────────────────────────────────
model_order = test_df.sort_values('test_rmse')['model'].tolist()

comparison = (
    midsem_df
    .merge(rv_df,   on='model', how='outer')
    .merge(test_df, on='model', how='outer')
)

# Reorder rows; CNN1D has no RV or test data — keep at bottom
core = comparison[comparison['model'].isin(model_order)].set_index('model').loc[model_order].reset_index()
rest = comparison[~comparison['model'].isin(model_order)]
comparison = pd.concat([core, rest], ignore_index=True)

comparison_path = os.path.join(OUT_DIR, 'final_model_comparison_fd001.csv')
comparison.to_csv(comparison_path, index=False)

print('E53 — Final model comparison:')
print(comparison.to_string(index=False))
print(f'\nSaved: {os.path.relpath(comparison_path, BASE)}')

E53 — Final model comparison:
             model  midsem_val_rmse  midsem_val_mae  midsem_val_r2  rv_mean_rmse  rv_std_rmse  rv_mean_mae  rv_mean_r2  rv_mean_bias  test_rmse  test_mae  test_r2  test_mean_bias
           XGBoost          12.4894          9.2675         0.9109       14.4297       2.0723      10.4897      0.8791       -1.7722    12.2459    9.0155   0.9066          0.4966
    DerivedOnlyMLP          13.1451          9.4204         0.9013       14.2308       0.9167      10.2354      0.8837       -0.3090    12.8295    9.5588   0.8975         -0.5532
               GRU          13.1605          9.7182         0.9010       14.4671       0.4212      11.2108      0.8800       -1.9374    13.2860    9.9167   0.8901          0.7607
MultiViewGRUFusion          12.0657          8.9406         0.9168       14.2787       1.6112      10.7053      0.8822       -3.5229    13.3782    9.9372   0.8885         -3.1971
             CNN1D          18.1509         14.0382         0.8118         

## Section 3 — E54: Research Question Evidence Table

Maps each dissertation research question to the primary evidence artefact and a one-sentence finding.

In [4]:
rq_rows = [
    {
        'rq_id':    'RQ1',
        'question': 'Can a multi-view deep learning model combining raw sensor sequences '
                    'with engineered degradation features predict RUL with competitive accuracy?',
        'primary_evidence': 'E01a, E05, E10',
        'finding':  'MultiViewGRUFusion achieved RMSE 12.07 on the mid-semester seed-42 '
                    'validation and mean RMSE 14.28 across three final-protocol splits, '
                    'placing it within 0.05 RMSE of DerivedOnlyMLP on validation. '
                    'On the official test it achieved RMSE 13.38, ranking fourth. '
                    'Fusion is competitive but not consistently superior.',
        'outcome':  'Partial — competitive but not consistently best',
    },
    {
        'rq_id':    'RQ2',
        'question': 'Does multi-view fusion consistently outperform single-view and '
                    'classical baselines across different engine-cohort splits?',
        'primary_evidence': 'E05, E06, E07a',
        'finding':  'Fusion did not achieve the lowest RMSE in any of the three '
                    'final-protocol splits. It outperformed each individual comparator '
                    'in only one of three splits. DerivedOnlyMLP had the lowest mean '
                    'RMSE across splits. Outcome C applies.',
        'outcome':  'No — Outcome C confirmed',
    },
    {
        'rq_id':    'RQ3',
        'question': 'Is model ranking sensitive to the choice of engine-level train/validation split?',
        'primary_evidence': 'E05, E10',
        'finding':  'The strongest model differed across splits in NB10 '
                    '(DerivedOnlyMLP for seed 21, XGBoost for seed 42, GRU for seed 84) '
                    'and between NB10 mean ranking and the NB11 official test '
                    '(XGBoost ranked first on test vs DerivedOnlyMLP on validation mean). '
                    'Outcome D provides additional descriptive evidence of cohort sensitivity.',
        'outcome':  'Yes — Outcome D confirmed',
    },
    {
        'rq_id':    'RQ4',
        'question': 'What is the contribution of the cycle_index feature to model performance?',
        'primary_evidence': 'E09',
        'finding':  'Removing cycle_index increased RMSE by 11.43% for XGBoost, '
                    '15.00% for MultiViewGRUFusion, and 16.32% for DerivedOnlyMLP '
                    'on the final-protocol seed-42 split. Lifecycle position '
                    'contributes materially to prediction. This does not constitute '
                    'target leakage because the current cycle number is observable at '
                    'inference time.',
        'outcome':  'Material contribution confirmed — not leakage',
    },
    {
        'rq_id':    'RQ5',
        'question': 'Can the model predictions be interpreted using explainability techniques?',
        'primary_evidence': 'E13–E24',
        'finding':  'SHAP analysis identified sensor_measurement_4_rmean and '
                    'cycle_index as the most influential features for XGBoost. '
                    'View masking showed that removing the derived degradation view '
                    'caused R² to drop to near zero, indicating that the engineered '
                    'features carry most of the predictive signal in the fusion model.',
        'outcome':  'Yes — XAI evidence supports interpretability claim',
    },
]

rq_df = pd.DataFrame(rq_rows)
rq_path = os.path.join(OUT_DIR, 'research_question_evidence_fd001.csv')
rq_df.to_csv(rq_path, index=False)

print('E54 — Research question evidence:')
for _, row in rq_df.iterrows():
    print(f"  {row['rq_id']}: {row['outcome']}")
print(f'\nSaved: {os.path.relpath(rq_path, BASE)}')

E54 — Research question evidence:
  RQ1: Partial — competitive but not consistently best
  RQ2: No — Outcome C confirmed
  RQ3: Yes — Outcome D confirmed
  RQ4: Material contribution confirmed — not leakage
  RQ5: Yes — XAI evidence supports interpretability claim

Saved: reports/final_summary/research_question_evidence_fd001.csv


## Section 4 — E55: Final Limitations Table

In [5]:
lim_rows = [
    {
        'limitation_id': 'L1',
        'area':          'Dataset scope',
        'description':   'Experiments are conducted on NASA C-MAPSS FD001 only — '
                         'a single operating condition, single fault mode, simulated dataset. '
                         'Generalisability to FD002–FD004 or real industrial datasets is not established.',
        'mitigant':      'Results are explicitly scoped to FD001. The multi-view framework '
                         'is designed to be dataset-agnostic in principle.',
    },
    {
        'limitation_id': 'L2',
        'area':          'Repeated-validation sample size',
        'description':   'Only three split seeds (21, 42, 84) were used for repeated validation. '
                         'Three configurations do not support formal statistical inference '
                         'about model ranking.',
        'mitigant':      'Findings are reported as descriptive stability evidence, '
                         'not statistically significant conclusions.',
    },
    {
        'limitation_id': 'L3',
        'area':          'Official test evaluation population',
        'description':   'The official test evaluates one endpoint prediction per engine, '
                         'not a full window trajectory. NB10 and NB11 RMSE values are '
                         'not like-for-like comparisons.',
        'mitigant':      'Results from each evaluation context are reported separately '
                         'with explicit population definitions.',
    },
    {
        'limitation_id': 'L4',
        'area':          'Fusion architecture search',
        'description':   'The MultiViewGRUFusion architecture was fixed from mid-semester '
                         'work without systematic hyperparameter search. A better fusion '
                         'design may outperform the single-view alternatives.',
        'mitigant':      'The frozen architecture ensures fair comparison across experiments. '
                         'The negative finding motivates future architecture exploration.',
    },
    {
        'limitation_id': 'L5',
        'area':          'Explainability scope',
        'description':   'SHAP analysis was applied to XGBoost only. Deep model '
                         'interpretability relied on view masking rather than gradient-based '
                         'attribution, which provides coarser feature-level explanations.',
        'mitigant':      'View masking is a model-agnostic technique consistent with '
                         'the multi-view framing. SHAP on XGBoost provides a complementary '
                         'feature importance signal.',
    },
    {
        'limitation_id': 'L6',
        'area':          'Evaluation metric',
        'description':   'RMSE and MAE treat early and late prediction errors symmetrically. '
                         'The asymmetric NASA scoring function, which penalises late '
                         'predictions more heavily, was not used as the primary metric.',
        'mitigant':      'The capped-RUL regression formulation is consistent with the '
                         'majority of published C-MAPSS work. Mean error is reported '
                         'separately to characterise directional bias.',
    },
]

lim_df = pd.DataFrame(lim_rows)
lim_path = os.path.join(OUT_DIR, 'final_limitations_fd001.csv')
lim_df.to_csv(lim_path, index=False)

print('E55 — Limitations:')
for _, row in lim_df.iterrows():
    print(f"  {row['limitation_id']} ({row['area']})")
print(f'\nSaved: {os.path.relpath(lim_path, BASE)}')

E55 — Limitations:
  L1 (Dataset scope)
  L2 (Repeated-validation sample size)
  L3 (Official test evaluation population)
  L4 (Fusion architecture search)
  L5 (Explainability scope)
  L6 (Evaluation metric)

Saved: reports/final_summary/final_limitations_fd001.csv


## Section 5 — E56: Final Claims and Qualifications

These are the dissertation's final stated claims, each qualified to match the actual evidence strength.

In [6]:
claims_rows = [
    {
        'claim_id':     'C1',
        'claim':        'A multi-view deep learning model combining raw sensor sequences '
                        'with engineered degradation features achieves competitive RUL '
                        'prediction accuracy on NASA C-MAPSS FD001.',
        'evidence':     'E01a, E05, E10',
        'qualification':'Competitive is defined relative to XGBoost, GRU and DerivedOnlyMLP '
                        'baselines on the same dataset and evaluation protocol. '
                        'Fusion achieved RMSE within 0.05 of DerivedOnlyMLP on mid-semester '
                        'validation but ranked fourth on the official test.',
        'strength':     'Supported with qualification',
    },
    {
        'claim_id':     'C2',
        'claim':        'Multi-view fusion does not consistently outperform the strongest '
                        'single-view or classical model across repeated engine-cohort splits.',
        'evidence':     'E05, E06, E07a, E10',
        'qualification':'Based on three split seeds and one official test evaluation on FD001. '
                        'Outcome C. Not a universal claim across datasets or architectures.',
        'strength':     'Supported',
    },
    {
        'claim_id':     'C3',
        'claim':        'Model ranking on NASA C-MAPSS FD001 is sensitive to the choice '
                        'of engine-level train/validation split.',
        'evidence':     'E05, E10',
        'qualification':'Descriptive evidence from three splits and one held-out test. '
                        'Outcome D. Not a statistically significant result.',
        'strength':     'Supported as descriptive finding',
    },
    {
        'claim_id':     'C4',
        'claim':        'The engineered degradation feature view carries most of the '
                        'predictive information in the current FD001 formulation.',
        'evidence':     'E06, E09, E21',
        'qualification':'Evidenced by DerivedOnlyMLP outperforming fusion on mean validation '
                        'RMSE, by view masking showing R² near zero without the derived view, '
                        'and by cycle_index ablation showing 11–16% RMSE degradation on removal. '
                        'Qualified to the current feature engineering and model design.',
        'strength':     'Supported',
    },
    {
        'claim_id':     'C5',
        'claim':        'SHAP analysis and view masking provide interpretable explanations '
                        'of model behaviour consistent with known turbofan degradation patterns.',
        'evidence':     'E13–E24',
        'qualification':'SHAP applied to XGBoost only. View masking applied to MultiViewGRUFusion. '
                        'Sensor_measurement_4_rmean and cycle_index identified as dominant '
                        'predictors, consistent with EDA findings in NB02.',
        'strength':     'Supported with scope qualification',
    },
    {
        'claim_id':     'C6',
        'claim':        'MultiViewGRUFusion shows a consistent tendency to under-predict RUL.',
        'evidence':     'E06, E10',
        'qualification':'Mean error of −3.52 cycles across repeated-validation splits '
                        'and −3.20 on the official test. Directionally consistent. '
                        'Business consequences depend on maintenance cost asymmetry, '
                        'which has not been modelled.',
        'strength':     'Supported',
    },
]

claims_df = pd.DataFrame(claims_rows)
claims_path = os.path.join(OUT_DIR, 'final_claims_and_qualifications_fd001.csv')
claims_df.to_csv(claims_path, index=False)

print('E56 — Final claims:')
for _, row in claims_df.iterrows():
    print(f"  {row['claim_id']}: {row['strength']}")
print(f'\nSaved: {os.path.relpath(claims_path, BASE)}')

E56 — Final claims:
  C1: Supported with qualification
  C2: Supported
  C3: Supported as descriptive finding
  C4: Supported
  C5: Supported with scope qualification
  C6: Supported

Saved: reports/final_summary/final_claims_and_qualifications_fd001.csv


### NB12 Completion Gate

- [x] No model training performed
- [x] No test data accessed
- [x] All numbers read from saved NB10 and NB11 artefacts
- [x] E53 — Final model comparison table generated
- [x] E54 — Research question evidence table generated
- [x] E55 — Final limitations table generated
- [x] E56 — Final claims and qualifications generated
- [x] All four artefacts saved to `reports/final_summary/`

### NB12 Generated Artefacts

In [7]:
print('=== NB12 GENERATED ARTEFACTS ===\n')
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.isfile(fpath):
        print(f'  reports/final_summary/{fname}  ({os.path.getsize(fpath):,} bytes)')

=== NB12 GENERATED ARTEFACTS ===

  reports/final_summary/.gitkeep  (0 bytes)
  reports/final_summary/evidence_ledger.md  (12,020 bytes)
  reports/final_summary/final_claims_and_qualifications_fd001.csv  (2,184 bytes)
  reports/final_summary/final_limitations_fd001.csv  (2,050 bytes)
  reports/final_summary/final_model_comparison_fd001.csv  (591 bytes)
  reports/final_summary/research_question_evidence_fd001.csv  (2,318 bytes)
